In [ ]:
import os
from PIL import Image

def split_grids_to_individual_images(input_base_dir, output_base_dir, grid_size=8):
    """
    Splits grid images into individual images for each step and organizes them by step.

    Args:
    - input_base_dir: Path to the base directory containing samples (e.g., sample1/, sample2/).
    - output_base_dir: Path to the base directory to save individual images by step.
    - grid_size: Number of rows and columns in the grid (default is 8x8 for 64 images per grid).
    """
    ref = 0 
    # Ensure the output directory exists
    os.makedirs(output_base_dir, exist_ok=True)

    # Get all sample directories (e.g., sample1/, sample2/, ...)
    sample_dirs = [d for d in os.listdir(input_base_dir) if os.path.isdir(os.path.join(input_base_dir, d))]
    print(sample_dirs)

    for sample_dir in sample_dirs:
        sample_path = os.path.join(input_base_dir, sample_dir)
        print(sorted(os.listdir(sample_path)))
        
        # Process all step images (e.g., steps0001.png, steps0002.png, ...)
        for grid_file in sorted(os.listdir(sample_path)):
            if grid_file.endswith(".png"):
                step_name = os.path.splitext(grid_file)[0]  # e.g., steps0001
                grid_path = os.path.join(sample_path, grid_file)

                # Output directory for this step (e.g., sample/step1, sample/step2, ...)
                step_number = int(step_name[-4:])  # Extract step number from filename
                step_dir = os.path.join(output_base_dir,f"step{step_number}")
                os.makedirs(step_dir,exist_ok=True)

                # Open the grid image
                grid_image = Image.open(grid_path)
                grid_width, grid_height = grid_image.size

                # Calculate the size of each individual image
                cell_width = grid_width // grid_size
                cell_height = grid_height // grid_size

                # Extract individual images from the grid
                count = 0
                for i in range(grid_size):
                    for j in range(grid_size):
                        left = j * cell_width
                        upper = i * cell_height
                        right = left + cell_width
                        lower = upper + cell_height

                        cropped_image = grid_image.crop((left, upper, right, lower))
                        cropped_image = cropped_image.resize((512,512))
                        output_filename = os.path.join(step_dir, f"{sample_dir}_img_{count:03d}_{ref}.png")
                        cropped_image.save(output_filename)
                        count += 1
                        ref +=1

                print(f"Processed {grid_file} -> {step_dir}")


input_base_dir = "FastDiffusion.py/out/unet_sm_test/"  # Replace with the base directory for your generated samples
output_base_dir = "FastDiffusion.py/out/unet_sm_single_images"    # Replace with the base directory to save individual images

split_grids_to_individual_images(input_base_dir, output_base_dir, grid_size=8)

In [ ]:
from cleanfid import fid
import os

def calculate_fid_per_step(generated_base_dir, val_dir):
    steps = sorted([d for d in os.listdir(generated_base_dir) if os.path.isdir(os.path.join(generated_base_dir, d))])

    for step in steps:
        step_dir = os.path.join(generated_base_dir, step)  # Path to generated images for this step
        print(f"Calculating FID for {step}...")

        fid_score = fid.compute_fid(step_dir, val_dir)
        print(f"FID for {step}: {fid_score}")

        
generated_base_dir = "FastDiffusion.py/out/unet_sm_single_images"  
val_dir = "FastDiffusion.py/data/AFHQ/train/test" 

calculate_fid_per_step(generated_base_dir, val_dir)
